In [19]:
from agents import AsyncOpenAI
from agents import (
    Agent, Runner, 
    #function_tool,
    OpenAIChatCompletionsModel,
    ModelSettings
    #set_tracking_disabled
)
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime


In [20]:
local_model = OpenAIChatCompletionsModel(
    model = "minimax-m2.7:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [21]:
# BAD instructions — too vague
bad_agent = Agent(
    name="Bad Agent",
    instructions="You are helpful.",
    model=local_model
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """,
    model=local_model
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Score: 6/10

**Issues:**
- No type hints (Python best practice for clarity)
- No docstring
- No input validation (e.g., if someone passes strings, it will silently concatenate)
- Limited to only built-in types

**Suggestions:**

```python
def add(x: float, y: float) -> float:
    """Add two numbers together.
    
    Args:
        x: First number
        y: Second number
    
    Returns:
        Sum of x and y
    
    Raises:
        TypeError: If x or y are not numeric types
    """
    if not isinstance(x, (int, float)) or not isinstance(y, (int, float)):
        raise TypeError(f"Both arguments must be numeric, got {type(x).__name__} and {type(y).__name__}")
    return x + y
```

**Verdict:** Fine as a quick example, but lacks production-readiness. The missing type hints and docstring would make this harder to maintain in a larger codebase.


In [24]:
# Creative agent — high temperature
creative_agent = Agent(
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    ),
    model=local_model
)

# Precise agent — low temperature
precise_agent = Agent(
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    ),
    model=local_model
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

OPENAI_API_KEY is not set, skipping trace export


 CREATIVE:
 

 PRECISE:
 

The **Badshahi Mosque** (Urdu: بادشاہی مسجد), located in Lahore, Pakistan, is a magnificent example of Mughal architecture and one of the most iconic landmarks in South Asia. Here’s a detailed description:

### **Historical Background**
- **Built:** 1671–1673 CE, during the reign of Emperor Aurang


OPENAI_API_KEY is not set, skipping trace export


In [25]:
local_model_1 = OpenAIChatCompletionsModel(
    model = "qwen3-vl:4b",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [26]:
# Define the output structure
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre (e.g., AI Research, Machine Learning, AI News, AI Ethics)")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# Create agent with structured output
reviewer = Agent(
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic. Analyze podcasts about artificial intelligence, "
        "machine learning, and related topics. Provide structured reviews and assess "
        "the technical depth and accessibility of each podcast."
    ),
    output_type=PodcastReview,
    model=local_model_1
)

result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# result.final_output is now a PodcastReview object, not a string!
review = result.final_output
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

OPENAI_API_KEY is not set, skipping trace export


Title:          Lex Fridman Podcast: A Deep Dive Analysis
Host:           Lex Fridman
Rating:         7.8/10
Genre:          AI/ML Technology Critique
Tech Level:     Advanced
Summary:        A weekly interview series featuring top experts in AI, ML, and tech, known for its broad scope and high-profile guests.
Recommended:    Yes


In [28]:
class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"] = Field(
        description="Product category"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently does the customer need this?"
    )
    price_range: Literal["budget", "mid", "premium"] = Field(
        description="Estimated price range based on description"
    )
    search_query: str = Field(
        description="Optimized search query for the product catalog"
    )
    confidence: float = Field(
        description="Confidence score 0.0-1.0"
    )


classifier = Agent(
    name="Product Classifier",
    instructions="""
    You classify customer product requests for an e-commerce platform.
    Analyze the customer's natural language description and extract:
    - What category the product falls into
    - How urgent the request seems
    - Estimated price range
    - An optimized search query
    - Your confidence in the classification
    
    Be accurate. If you're unsure, set confidence low.
    """,
    output_type=ProductClassification,
    model_settings=ModelSettings(temperature=0.1),
    model=local_model_1
)

# Test with different customer requests
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

for req in requests:
    result = await Runner.run(classifier, req)
    c = result.final_output
    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")

OPENAI_API_KEY is not set, skipping trace export



Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: high | Price: budget
   Search: 'school laptop under $500' | Confidence: 85%


OPENAI_API_KEY is not set, skipping trace export



Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: medium | Price: mid
   Search: 'birthday gift novel' | Confidence: 9000%


OPENAI_API_KEY is not set, skipping trace export



Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: mid
   Search: 'phone charger replacement' | Confidence: 95%


In [30]:
def dynamic_instructions(context, agent):
    """Instructions that change based on current state."""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    return f"""
    You are a scheduling assistant.
    Current date/time: {current_time}
    
    Rules:
    - If it's morning (before 12pm), suggest productive tasks.
    - If it's afternoon, suggest meetings and collaborations.
    - If it's evening, suggest winding down activities.
    - Always be aware of the current time in your suggestions.
    """


scheduler = Agent(
    name="Smart Scheduler",
    instructions=dynamic_instructions,  # <-- Pass function, not string!,
    model=local_model
)

result = await Runner.run(scheduler, "What should I do right now?")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


In [32]:

class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")


ticket_analyzer = Agent(
    name="Ticket Analyzer",
    instructions="""
    You are an AI support ticket analyzer for a SaaS company.
    
    PRIORITY RULES:
    - P0-critical: Service down, data loss, security breach
    - P1-high: Feature broken, payment issues, angry customer
    - P2-medium: Bug reports, feature requests, how-to questions
    - P3-low: General feedback, suggestions, non-urgent inquiries
    
    DEPARTMENT ROUTING:
    - billing: Payment, subscription, invoice, refund
    - technical: Bugs, errors, API issues, integrations
    - sales: Pricing, enterprise plans, demos, upgrades
    - general: Everything else
    
    Write the suggested_response in a professional, empathetic tone.
    """,
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.2),
    model=local_model_1
)

# Simulate incoming tickets
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)
    t = result.final_output
    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export



Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: negative
Priority:  P0-critical
Route to:  technical
Summary:   API 500 errors for 2 hours causing production downtime
Response:  Dear Customer,

We have confirmed that your API has been returning 500 errors for 2 hours, resulting in your production being down. We sincerely apolo...

Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: negative
Priority:  P1-high
Route to:  billing
Summary:   Customer reported double charge this month
Response:  Dear Customer,

We are sorry to hear about the double charge you received this month. This is a billing error, and we understand how frustrating it mu...


OPENAI_API_KEY is not set, skipping trace export



Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: positive
Priority:  P2-medium
Route to:  general
Summary:   Feature request for dark mode
Response:  Thanks for the kind words! We really appreciate your feedback. Dark mode is a feature we've been considering for our users, and we'll definitely keep ...


OPENAI_API_KEY is not set, skipping trace export
